In [ ]:
"""
FASE 4: M(d) en escala log-log (conservadores, agresivos y combinados)
======================================================================
Tesis: Complejidad e informacion en un minority game con opcion de abstencion

Que hace este codigo
--------------------
NO vuelve a simular. Lee los CSV "todas_las_cadenas" que genero el codigo
Fase4_conservadores_agresivos (carpeta resultados_fase4/csv) y, para cada
curva (tipo, N, fraccion):

  1. Promedia M(d) sobre las 100 cadenas, para d = 1, ..., 100.
  2. Quita los puntos con M(d) = 0, porque log(0) no existe.
     (Es el UNICO filtro, por indicacion del Dr. Mansilla.)
  3. Calcula log10(d) y log10(M(d)).
  4. Ajusta una recta por minimos cuadrados con TODOS los puntos que quedan:
         log10 M(d) = b + pendiente * log10(d)
     Si M(d) = A / d^alpha, entonces pendiente = -alpha y b = log10(A).
  5. Guarda la PENDIENTE y la R^2 de cada curva en un CSV.

Graficas: UNA figura por cada N, con una cuadricula de paneles:
  filas    = fracciones (10%, 25%, 30%)
  columnas = solo conservadores, solo agresivos, combinados
Cada panel tiene log10 M(d) contra log10 d, la recta ajustada, y su
pendiente y R^2. Es el mismo procedimiento de las fases 2 y 3.

Notacion: gamma = 3^m / N es el parametro de control (no va en las
graficas; se guarda en el CSV). alpha es el exponente de la ley de
potencias M(d) = A / d^alpha.
"""

import os
import numpy as np
import matplotlib.pyplot as plt


# =====================================================================
# OPCIONES  <-- AQUI SE DECIDE SI SE GUARDAN LOS ARCHIVOS
# =====================================================================
# Las graficas SIEMPRE se muestran en el notebook.
# True  = ademas se guardan como PNG en CARPETA_SALIDA
# False = solo se muestran, no se guarda ningun PNG
DESCARGAR_GRAFICAS = True

# True  = se guarda un CSV con la pendiente y la R^2 de todas las curvas
GUARDAR_CSV        = True

# Donde estan los CSV que genero Fase4_conservadores_agresivos
CARPETA_CSV    = os.path.join("resultados_fase4", "csv")
CARPETA_SALIDA = "resultados_fase4_loglog"


# =====================================================================
# QUE SE ANALIZA  (deben coincidir con lo que se corrio en la fase 4)
# =====================================================================
LISTA_N    = [50, 100, 125, 151, 175, 200]
FRACCIONES = [0.10, 0.25, 0.30]

TIPOS = [
    ("conservadores", "Solo conservadores"),
    ("agresivos",     "Solo agresivos"),
    ("combinados",    "Combinados"),
]

M, S, K, T, N_CADENAS = 3, 2, 3, 6000, 100


def pct(fraccion):
    """0.25 -> 25 (para nombres y titulos)."""
    return int(round(fraccion * 100))


def numero_de_agentes(fraccion, N):
    """Cuantos agentes de UN tipo (mismo redondeo que la simulacion)."""
    return int(np.floor(fraccion * N + 0.5))


# =====================================================================
# LECTURA Y PROMEDIO
# =====================================================================
def leer_promedio(archivo):
    """Lee el CSV de todas las cadenas y devuelve d y el promedio de M(d)."""
    tabla = np.loadtxt(archivo, delimiter=",", comments="#")
    valores = tabla[:, 1:]                        # se quita la columna semilla
    distancias = np.arange(1, valores.shape[1] + 1)
    return distancias, valores.mean(axis=0), valores.shape[0]


# =====================================================================
# AJUSTE LINEAL EN ESCALA LOG-LOG  (igual que en las fases 2 y 3)
# =====================================================================
def ajuste_loglog(d, Md):
    """Quita los ceros, pasa a log10 y ajusta una recta."""
    validos = Md > 0                      # UNICO filtro: fuera M(d) = 0
    x = np.log10(d[validos])
    y = np.log10(Md[validos])

    pendiente, ordenada = np.polyfit(x, y, 1)
    y_ajuste = ordenada + pendiente * x

    ss_res = np.sum((y - y_ajuste) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot

    return {
        "pendiente": pendiente, "ordenada": ordenada, "r2": r2,
        "x": x, "y": y, "y_ajuste": y_ajuste,
        "n_puntos": int(validos.sum()), "n_ceros": int((~validos).sum()),
    }


# =====================================================================
# GRAFICA: UNA FIGURA POR N
# =====================================================================
GRIS = "#52514e"
TINTA = "#0b0b0b"
REJILLA = "#e5e4df"
# mismo color y marcador de cada tipo que en las graficas de la fase 4
ESTILO = {
    "conservadores": {"color": "#2a78d6", "marker": "o"},
    "agresivos":     {"color": "#eb6834", "marker": "s"},
    "combinados":    {"color": "#1baf7a", "marker": "^"},
}

plt.rcParams.update({
    "font.size": 10,
    "axes.edgecolor": GRIS,
    "axes.labelcolor": TINTA,
    "xtick.color": GRIS,
    "ytick.color": GRIS,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


def graficar_N(N, ajustes, guardar):
    filas, columnas = len(FRACCIONES), len(TIPOS)
    fig, ejes = plt.subplots(filas, columnas,
                             figsize=(4.6 * columnas, 3.4 * filas),
                             sharex=True, sharey=True, squeeze=False)

    for i, fraccion in enumerate(FRACCIONES):
        for j, (clave, nombre) in enumerate(TIPOS):
            ax = ejes[i, j]
            res = ajustes.get((clave, N, fraccion))
            if res is None:
                ax.set_visible(False)
                continue
            est = ESTILO[clave]
            ax.plot(res["x"], res["y"], est["marker"], color=est["color"],
                    ms=3.5, label="log$_{10}$ M(d), promedio")
            ax.plot(res["x"], res["y_ajuste"], color=TINTA, lw=1.6,
                    label="Ajuste lineal")
            ax.text(0.97, 0.95,
                    f"pendiente = {res['pendiente']:.4f}\n"
                    f"R$^2$ = {res['r2']:.4f}",
                    transform=ax.transAxes, ha="right", va="top",
                    fontsize=9, color=TINTA,
                    bbox=dict(boxstyle="round,pad=0.3", fc="white",
                              ec=REJILLA))
            ax.grid(color=REJILLA, lw=0.7)

            n_tipo = numero_de_agentes(fraccion, N)
            agentes = (f"{n_tipo} + {n_tipo}" if clave == "combinados"
                       else f"{n_tipo}")
            ax.set_title(f"{nombre}, {pct(fraccion)}%  ({agentes} agentes)",
                         loc="left", fontsize=9.5, color=TINTA)
            if i == filas - 1:
                ax.set_xlabel("log$_{10}$ d")
            if j == 0:
                ax.set_ylabel("log$_{10}$ M(d)  [bits]")

    fig.suptitle(f"M(d) en escala log-log — conservadores, agresivos y "
                 f"combinados\nN={N}, m={M}, s={S}, K={K}, T={T}, con puntaje, "
                 f"{N_CADENAS} semillas",
                 x=0.01, y=0.995, ha="left", fontsize=12, color=TINTA)
    manijas = [
        plt.Line2D([], [], color=ESTILO[c]["color"], marker=ESTILO[c]["marker"],
                   lw=0, ms=5) for c, _ in TIPOS
    ] + [plt.Line2D([], [], color=TINTA, lw=1.6)]
    etiquetas = [n for _, n in TIPOS] + ["Ajuste lineal"]
    fig.legend(manijas, etiquetas, loc="lower center", ncol=4, frameon=False,
               fontsize=9.5)
    fig.tight_layout(rect=[0, 0.035, 1, 0.95])

    if guardar:
        os.makedirs(CARPETA_SALIDA, exist_ok=True)
        ruta = os.path.join(CARPETA_SALIDA, f"Fase4_loglog_N{N}.png")
        fig.savefig(ruta, dpi=200)
        print(f"  Guardado: {ruta}")
    plt.show()
    plt.close(fig)


# =====================================================================
# PROGRAMA PRINCIPAL
# =====================================================================
if __name__ == "__main__":

    ajustes = {}
    filas_csv = []

    for N in LISTA_N:
        print("\n" + "=" * 70)
        print(f"  N = {N}")
        print("=" * 70)
        print(f"  {'caso':20s} {'%':>3s} {'ceros':>6s} {'puntos':>7s} "
              f"{'pendiente':>10s} {'R^2':>8s}")

        for fraccion in FRACCIONES:
            for clave, nombre in TIPOS:
                archivo = os.path.join(
                    CARPETA_CSV,
                    f"Fase4_Md_{clave}_N{N}_f{pct(fraccion)}"
                    f"_todas_las_cadenas.csv")
                if not os.path.exists(archivo):
                    print(f"  FALTA {archivo}; se omite.")
                    continue
                d, Md, n_cad = leer_promedio(archivo)
                res = ajuste_loglog(d, Md)
                ajustes[(clave, N, fraccion)] = res

                print(f"  {nombre:20s} {pct(fraccion):>3d} {res['n_ceros']:>6d} "
                      f"{res['n_puntos']:>7d} {res['pendiente']:>10.4f} "
                      f"{res['r2']:>8.4f}")

                n_tipo = numero_de_agentes(fraccion, N)
                filas_csv.append([
                    clave, N, K ** M / N, pct(fraccion),
                    n_tipo if clave in ("conservadores", "combinados") else 0,
                    n_tipo if clave in ("agresivos", "combinados") else 0,
                    res["n_ceros"], res["n_puntos"],
                    res["pendiente"], res["r2"], res["ordenada"],
                ])

        graficar_N(N, ajustes, DESCARGAR_GRAFICAS)

    if GUARDAR_CSV and filas_csv:
        os.makedirs(CARPETA_SALIDA, exist_ok=True)
        ruta = os.path.join(CARPETA_SALIDA, "Fase4_ajustes_loglog.csv")
        with open(ruta, "w", encoding="utf-8") as f:
            f.write("# Ajuste lineal de log10 M(d) vs log10 d, d = 1..100, "
                    "promedio de 100 cadenas\n")
            f.write(f"# Fase 4, m={M} s={S} K={K} T={T} con puntaje; "
                    "unico filtro: se quitan los M(d) = 0\n")
            f.write("tipo,N,gamma,fraccion_pct,conservadores,agresivos,"
                    "ceros_eliminados,puntos_usados,pendiente,R2,"
                    "ordenada_log10\n")
            for t, N, g, p, nc, na, c, n, pe, r2, b in filas_csv:
                f.write(f"{t},{N},{g:.6f},{p},{nc},{na},{c},{n},"
                        f"{pe:.6f},{r2:.6f},{b:.6f}\n")
        print(f"\n  Guardado: {ruta}")
